## Exponential Smoothing Baseline

In [ ]:
from pathlib import Path
import polars as pl

from datetime import timedelta
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import polars as pl
from tqdm import tqdm

from statsmodels.tsa.holtwinters import ExponentialSmoothing

from constants import (
    UCI_CLIENT_SITES_TO_VALIDATE,
    UCI_DATA_FREQUENCY_MINUTES,
    UCI_VALIDATION_START,
    UCI_VALIDATION_WINDOW,
)

In [ ]:
# Constants

UCI_DATA_DIR = Path("../../data/uci")
UCI_DATA_FILE_NAME = "preprocessed.pq"
UCI_DATA_PATH = UCI_DATA_DIR / UCI_DATA_FILE_NAME

N_VALIDATION_FOLDS = 10

RESULTS_OUTPUT_DIR = Path("../../results/uci/ets")

In [ ]:
# Load preprocessed data

uci_df = pl.read_parquet(UCI_DATA_PATH)
UCI_CLIENT_DF = uci_df.filter(pl.col("client").is_in(UCI_CLIENT_SITES_TO_VALIDATE))

In [ ]:
for client in tqdm(UCI_CLIENT_SITES_TO_VALIDATE):
    client_results_dir = RESULTS_OUTPUT_DIR / client
    client_results_dir.mkdir(parents=True, exist_ok=True)
    
    client_df = UCI_CLIENT_DF.filter(pl.col("client") == client)
    
    for k in range(N_VALIDATION_FOLDS):
        val_start = UCI_VALIDATION_START + k * UCI_VALIDATION_WINDOW
        val_end = val_start + UCI_VALIDATION_WINDOW
        train_df = (
            client_df
            .filter(pl.col("timestamp").lt(val_start))
            .select(pl.col("timestamp"), pl.col("demand"))
            .sort(by="timestamp")
        )
        valid_df = (
            client_df
            .filter(pl.col("timestamp").is_between(val_start, val_end, closed="left"))
            .select(pl.col("timestamp"), pl.col("demand"))
            .sort(by="timestamp")
        )

        # Train exponential smoothing model with daily seasonality
        train_ts = train_df.to_pandas().set_index("timestamp")
        ets_model = ExponentialSmoothing(
            endog=train_ts,
            trend=None,
            damped_trend=False,
            seasonal="add",
            seasonal_periods=int(7 * 24 * 60 / UCI_DATA_FREQUENCY_MINUTES),
            freq=f"{UCI_DATA_FREQUENCY_MINUTES}min"
        )
        trained_ets_model = ets_model.fit()

        # Get fitted values.
        train_start = val_start - timedelta(days=30)
        fitted_values = trained_ets_model.predict(start=train_start, end=val_end)
        fitted_values_df = pl.DataFrame({"timestamp": fitted_values.index, "forecast": fitted_values})
        
        # Plot fitted values and save.
        time_unit = train_df["timestamp"].dtype.time_unit
        fitted_values_df = (
            train_df.select(pl.col("timestamp").dt.cast_time_unit(time_unit), pl.col("demand"))
            .join(
                other=fitted_values_df.select(pl.col("timestamp").dt.cast_time_unit(time_unit), pl.col("forecast")),
                on="timestamp",
                how="inner"
            )
        )
        
        fig, ax = plt.subplots()
        ax.plot(fitted_values_df["timestamp"], fitted_values_df["demand"], color="black", lw=2, label="Actual")
        ax.plot(fitted_values_df["timestamp"], fitted_values_df["forecast"], color="#0072B2", lw=2, label="Fitted")
        
        ax.legend(loc=1)
        ax.grid(True, which="major", c="grey", ls="--", lw=1, alpha=0.2)
        ax.set(ylabel="Load (kWh)", title=f"Electricity Load Fitted Values for Site {client}, Fold {k}")
        
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax.tick_params(axis='x', labelrotation=45)
        
        fig.tight_layout()
        plt_save_path = client_results_dir / f"fitted_values_{client}_fold_{k}.png"
        plt.savefig(plt_save_path, dpi=300)
        plt.close(fig);
        
        # Forecast
        y_hat = trained_ets_model.predict(start=val_start, end=val_end)
        y_hat_df = pl.DataFrame({"timestamp": y_hat.index, "forecast": y_hat})
        
        # Save forecasts
        time_unit = valid_df["timestamp"].dtype.time_unit
        forecasts_df = (
            valid_df.select(pl.col("timestamp").dt.cast_time_unit(time_unit), pl.col("demand"))
            .join(
                other=y_hat_df.select(pl.col("timestamp").dt.cast_time_unit(time_unit), pl.col("forecast")),
                on="timestamp",
                how="left"
            )
        )
        forecast_output_path = client_results_dir / f"forecasts_{client}_fold_{k}.pq"
        forecasts_df.to_pandas().to_parquet(forecast_output_path)

        # Plot forecasts and save
        fig, ax = plt.subplots()
        
        ax.plot(forecasts_df["timestamp"], forecasts_df["demand"], color="black", lw=2, label="Actual")
        ax.plot(forecasts_df["timestamp"], forecasts_df["forecast"], color="#0072B2", lw=2, label="Forecast")
        
        ax.legend(loc=1)
        ax.grid(True, which="major", c="grey", ls="--", lw=1, alpha=0.2)
        ax.set(ylabel="Load (kWh)", title=f"Electricity Load Forecasts for Site {client}, Fold {k}")
        
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax.tick_params(axis='x', labelrotation=45)
        
        fig.tight_layout()
        plt_save_path = client_results_dir / f"forecasts_{client}_fold_{k}.png"
        plt.savefig(plt_save_path, dpi=300)
        plt.close(fig);